# MMDL-NAI Representation and Classification

In [11]:
# ============================
# MMDL-NAI 
# Load from Pretrained MMDL-NAI model
# ----------------------------

from tensorflow.keras.models import model_from_json
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import numpy as np


with open("MMDL_NAI.json", "r") as f:
    loaded_model = model_from_json(f.read())

loaded_model.load_weights("MMDL_NAI.weights.h5")

loaded_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001, epsilon=1e-7),
    metrics=['accuracy']
)

# ----------------------------
# pre-compute split (สำคัญมาก)
# ----------------------------
splits = list(skf.split(X_train, y_train))

sepscores_ind = []

scaler = StandardScaler()
scaler.fit(X_train[tr_idx])

X_test_fold = scaler.transform(X_test)

    # reshape แบบ memory-friendly (ไม่ copy เกินจำเป็น)
X_test_fold = np.expand_dims(X_test_fold, axis=1)

    # predict แบบลด memory spike
y_score = loaded_model.predict(X_test_fold, batch_size=32, verbose=0)

y_class = np.argmax(y_score, axis=1)
y_score_pos = y_score[:, 1]

acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_class), y_class, y_test)

fpr, tpr, _ = roc_curve(y_test, y_score_pos)
roc_auc = auc(fpr, tpr)

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score_pos)
au_pr = auc(recall_curve, precision_curve)

sepscores_ind.append([
        acc, precision, npv, sensitivity, specificity,
        mcc, f1, roc_auc, au_pr, tp, tn, fp, fn
    ])

print(f"MCC={mcc:.4f}, AUC={roc_auc:.4f}, AU_PR={au_pr:.4f}")

MCC=0.9066, AUC=0.9727, AU_PR=0.9734


2026-06-19 07:22:21.170235: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [ ]:
# ============================
# MMDL-NAI 
# Run all training and test
# ============================

# ============================

# ============================
import os
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# ============================
# Imports
# ============================
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

tf.config.experimental.enable_op_determinism()

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import Dense, Dropout, Conv1D, AveragePooling1D, Flatten
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# Utility functions
# -----------------------------
def categorical_probas_to_classes(p):
    return np.argmax(p, axis=1)

def calculate_performance(test_num, pred_y, labels):
    tp = fp = tn = fn = 0
    for i in range(test_num):
        if labels[i] == 1:
            if pred_y[i] == 1: tp += 1
            else: fn += 1
        else:
            if pred_y[i] == 0: tn += 1
            else: fp += 1

    acc = (tp + tn) / test_num
    precision = tp / (tp + fp + 1e-6)
    npv = tn / (tn + fn + 1e-6)
    sensitivity = tp / (tp + fn + 1e-6)
    specificity = tn / (tn + fp + 1e-6)
    mcc = (tp * tn - fp * fn) / (
        math.sqrt((tp + fp)*(tp + fn)*(tn + fp)*(tn + fn)) + 1e-6
    )
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)

    return acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn

# -----------------------------
# Model
# -----------------------------
def get_MMDL_NAI_model(input_shape):

    model = Sequential()

    model.add(Conv1D(
        filters=32, # 32
        kernel_size=3,
        padding='same',
        activation='relu',
        input_shape=input_shape
    ))

    model.add(AveragePooling1D(pool_size=2, strides=1, padding="same"))

    model.add(Conv1D(filters=16, kernel_size=3, padding='same', activation='relu')) #16

    model.add(AveragePooling1D(pool_size=2, strides=1, padding="same"))

    model.add(Flatten())
    model.add(Dense(5, activation='relu')) #5
    model.add(Dense(2, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001, epsilon=1e-7),
        metrics=['accuracy']
    )

    return model

# -----------------------------
# Load data

train_path = 'multimodal_train.csv'
test_path  = 'multimodal_test.csv'

y_train = [0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1]
 
y_test = [1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1]

y_train = np.array(y_train)
y_test  = np.array(y_test)


input_shape = (1, X_train.shape[1])

# -----------------------------
# Cross-validation
# -----------------------------
save_dir = f'./MMDL_NAI_seed{SEED}/'
os.makedirs(save_dir, exist_ok=True)

sepscores_cnn = []

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
callbacks = [reduce_lr, early_stop]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

oof_train_prob = np.zeros(len(X_train))
test_prob_folds = []

for i, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

    print(f"\n===== Fold {i} =====")
    tf.keras.backend.clear_session()

    # =============================
    #  CLEAN SCALER PER FOLD
    # =============================
    scaler = StandardScaler()

    X_tr_raw  = X_train[tr_idx]
    X_val_raw = X_train[val_idx]

    X_tr  = scaler.fit_transform(X_tr_raw)
    X_val = scaler.transform(X_val_raw)
    X_test_fold = scaler.transform(X_test)
    #X_tr  = X_tr_raw
    #X_val = X_val_raw
    #X_test_fold = X_test
    # reshape หลัง scale
    X_tr  = X_tr.reshape(X_tr.shape[0], 1, X_tr.shape[1])
    X_val = X_val.reshape(X_val.shape[0], 1, X_val.shape[1])
    X_test_fold = X_test_fold.reshape(X_test_fold.shape[0], 1, X_test_fold.shape[1])

    model = get_MMDL_NAI_model((1, X_tr.shape[2]))

    y_tr  = y_train[tr_idx]
    y_val = y_train[val_idx]

    y_tr_cat  = to_categorical(y_tr, 2)
    y_val_cat = to_categorical(y_val, 2)

    model.fit(
        X_tr,
        y_tr_cat,
        validation_data=(X_val, y_val_cat),
        epochs=50,
        batch_size=8,
        callbacks=callbacks,
        verbose=0
    )

    y_val_prob_full = model.predict(X_val)
    y_val_class = np.argmax(y_val_prob_full, axis=1)
    y_val_prob = y_val_prob_full[:, 1]

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_val_class), y_val_class, y_val)

    fpr, tpr, _ = roc_curve(y_val, y_val_prob)
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(y_val, y_val_prob)
    au_pr = auc(recall_curve, precision_curve)

    sepscores_cnn.append([
        acc, precision, npv, sensitivity, specificity,
        mcc, f1, roc_auc, au_pr, tp, tn, fp, fn
    ])

    print(f"Fold {i} CV: MCC={mcc:.4f}, AUC={roc_auc:.4f}, AU_PR={au_pr:.4f}")

    oof_train_prob[val_idx] = y_val_prob

    #  test prob using fold scaler
    test_prob = model.predict(X_test_fold)[:, 1]
    test_prob_folds.append(test_prob)

    with open(f"{save_dir}/MMDL_NAI_fold{i}.json", "w") as f:
        f.write(model.to_json())

    model.save_weights(f"{save_dir}/MMDL_NAI_fold{i}.weights.h5")

# -----------------------------
# Independent test (ใช้ scaler ของแต่ละ fold)
# -----------------------------
sepscores_ind = []

for i in range(len(sepscores_cnn)):

    scaler = StandardScaler()
    tr_idx, _ = list(skf.split(X_train, y_train))[i]
    scaler.fit(X_train[tr_idx])

    X_test_fold = scaler.transform(X_test)
    #X_test_fold = X_test
    X_test_fold = X_test_fold.reshape(X_test_fold.shape[0], 1, X_test_fold.shape[1])

    with open(f"{save_dir}/MMDL_NAI_fold{i}.json", "r") as f:
        loaded_model = model_from_json(f.read())

    loaded_model.load_weights(f"{save_dir}/MMDL_NAI_fold{i}.weights.h5")
    loaded_model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001, epsilon=1e-7),
        metrics=['accuracy']
    )

    y_score = loaded_model.predict(X_test_fold)
    y_class = np.argmax(y_score, axis=1)
    y_score = y_score[:, 1]

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_class), y_class, y_test)

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
    au_pr = auc(recall_curve, precision_curve)

    sepscores_ind.append(
        [acc, precision, npv, sensitivity, specificity,
         mcc, f1, roc_auc, au_pr, tp, tn, fp, fn]
    )

    print(f"Fold {i} test: MCC={mcc:.4f}, AUC={roc_auc:.4f}, AU_PR={au_pr:.4f}")

# -----------------------------
# Save results
# -----------------------------
cv_df = pd.DataFrame(
    sepscores_cnn,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
cv_df.to_csv(f"{save_dir}/results_CV.csv", index=False)

test_df_result = pd.DataFrame(
    sepscores_ind,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
test_df_result.to_csv(f"{save_dir}/results_Independent.csv", index=False)

